# UMAP Pipeline — Zeng / Zhuang / ISD

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import os

In [ ]:
# --- Hardcoded Palette ---
CELL_TYPE_PALETTE = {
    'ABCs': '#023FA5', 'Astrocytes': '#7D87B9', 'Astroependymal': '#BEC1D4',
    'BAMs': '#D6BCC0', 'Bergmann': '#BB7784', 'Choroid-Plexus': '#8E063B',
    'ECs': '#4A6FE3', 'Ependymal': '#8595E1', 'Immune-Other': '#B5BBE3',
    'Microglia': '#E6AFB9', 'Neurons-Dopa': '#E07B91', 'Neurons-Gaba': '#D33F6A',
    'Neurons-Glut': '#11C638', 'Neurons-Glyc-Gaba': '#8DD593',
    'Neurons-Granule-Immature': '#C6DEC7', 'Neurons-Other': '#EAD3C6',
    'OECs': '#F0B98D', 'OPCs': '#EF9708', 'Oligodendrocytes': '#0FCFC0',
    'Pericytes': '#9CDED6', 'SMCs': '#D5EAE7', 'Tanycytes': '#F3E1EB',
    'Undefined': '#F6C4E1', 'Unknown': '#F6C4E1', 'VLMCs': '#F79CD4'
}

In [ ]:
def add_concept_embeddings(adata, embedding_path, name="Dataset", cell_ids_removals=[]):
    """Load concept embeddings and add them to adata.obsm."""
    cell_ids = np.load(f"{embedding_path}/cell_ids.npy", allow_pickle=True).astype(str)
    emb_mean = np.load(f"{embedding_path}/cell_embs_mean.npy")
    emb_cls  = np.load(f"{embedding_path}/cell_embs_cls.npy")

    df_mean = pd.DataFrame(emb_mean, index=cell_ids)
    df_cls  = pd.DataFrame(emb_cls,  index=cell_ids)

    print(adata.obs_names)
    for removal in cell_ids_removals:
        adata.obs_names = adata.obs_names.str.replace(removal, "")
    print("after removal")
    print(adata.obs_names)

    # Filter adata to keep only cells with embeddings
    common_cells = adata.obs_names.intersection(cell_ids)
    print(f"   {name}: {len(common_cells):,}/{adata.n_obs:,} cells have embeddings")
    adata = adata[common_cells].copy()

    # Align embeddings
    adata.obsm["concept_mean_embedding"] = df_mean.loc[adata.obs_names].to_numpy()
    adata.obsm["concept_cls_embedding"]  = df_cls.loc[adata.obs_names].to_numpy()

    return adata

In [ ]:
def compute_and_plot_umap(adata, embedding_key, color_keys, output_dir, palette_map=None):
    print(f"Computing Neighbors and UMAP for embedding: {embedding_key}...")

    sc.pp.neighbors(adata, n_neighbors=30, use_rep=embedding_key)
    sc.tl.umap(adata, min_dist=0.3, random_state=0)

    os.makedirs(output_dir, exist_ok=True)

    for color in color_keys:
        title = f"UMAP {embedding_key} colored by {color}"
        out_png = os.path.join(output_dir, f"umap_{embedding_key}_{color}.png")

        use_palette = None
        if palette_map:
            cats = adata.obs[color].unique()
            if any(c in palette_map for c in cats):
                use_palette = palette_map

        fig = sc.pl.umap(
            adata,
            color=color,
            title=title,
            palette=use_palette,
            frameon=False,
            show=False,
            return_fig=True
        )

        fig.savefig(out_png, dpi=300, bbox_inches="tight")
        plt.show()
        plt.close(fig)
        print(f"Saved: {out_png}")

## Config

In [ ]:
MODEL_DIR  = "/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/concept_embeddings/models/small_weighted__hgld4ax1_last"
OUTPUT_DIR = "/p/scratch/cjinm16/dipippo1/scConcept/umaps"

batch_key            = "symphony_dataset"
label_key            = "cell_type"
embedding_obsm_keys  = ["concept_cls_embedding"]

## Load raw datasets

In [ ]:
zeng   = sc.read_h5ad("/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/Zeng.h5ad")
zhuang = sc.read_h5ad("/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/Zhuang-ABCA-1.h5ad")
isd    = sc.read_h5ad("/p/project1/hai_fzj_bda/spitzer2/point_transformer/data/raw/concept_embeddings/hvg_isd_normed.h5ad")

zeng.X   = None
zhuang.X = None
isd.X    = None

## Add embeddings

In [ ]:
zeng_embedded   = add_concept_embeddings(zeng,   os.path.join(MODEL_DIR, "zeng"),   name="zeng",   cell_ids_removals=["-Zeng"])
zhuang_embedded = add_concept_embeddings(zhuang, os.path.join(MODEL_DIR, "zhuang"), name="zhuang", cell_ids_removals=["-Zhuang-ABCA-1"])
isd_embedded    = add_concept_embeddings(isd,    os.path.join(MODEL_DIR, "isd"),    name="isd")

## Load annotations + merge

In [ ]:
annotated_isd    = sc.read_h5ad("/p/project1/hai_fzj_bda/salg1/point_transformer/data/processed/annotated_isd.h5ad")
annotated_zhuang = sc.read_h5ad("/p/project1/hai_fzj_bda/salg1/point_transformer/data/processed/annotated_zhuang.h5ad")
annotated_zeng   = sc.read_h5ad("/p/project1/hai_fzj_bda/salg1/point_transformer/data/processed/annotated_zeng.h5ad")

In [ ]:
annotated_zeng.obs["cell_type"]   = annotated_zeng.obs.cell_type_mmc_raw
annotated_zhuang.obs["cell_type"] = annotated_zhuang.obs.cell_type_mmc_raw
annotated_isd.obs["cell_type"]    = annotated_isd.obs.cell_type_mmc_raw_revised

zeng_embedded.obs["cell_id"]   = zeng_embedded.obs.index
zhuang_embedded.obs["cell_id"] = zhuang_embedded.obs.index
isd_embedded.obs["cell_id"]    = isd_embedded.obs.index

annotated_zeng.obs["cell_id"]   = annotated_zeng.obs.cell_id.str.replace("-Zeng", "")
annotated_zhuang.obs["cell_id"] = annotated_zhuang.obs.cell_id.str.replace("-Zhuang-ABCA-1-Zhuang", "")

In [ ]:
zhuang_embedded.obs = pd.merge(
    zhuang_embedded.obs,
    annotated_zhuang.obs[["cell_id", "cell_type"]],
    on="cell_id", how="left"
)

zeng_embedded.obs = pd.merge(
    zeng_embedded.obs,
    annotated_zeng.obs[["cell_id", "cell_type"]],
    on="cell_id", how="left"
)

isd_embedded.obs = pd.merge(
    isd_embedded.obs,
    annotated_isd.obs[["cell_id", "cell_type"]],
    on="cell_id", how="left"
)

## Subsample + Concat

In [ ]:
sc.pp.subsample(zeng_embedded,   n_obs=50000, random_state=0)
sc.pp.subsample(zhuang_embedded, n_obs=50000, random_state=0)

In [ ]:
adata_ref = ad.concat(
    {"ZHUANG": zhuang_embedded, "ZENG": zeng_embedded, "ISD": isd_embedded},
    label="symphony_dataset"
)

## Prepare label_key

In [ ]:
adata_ref.obs[label_key] = adata_ref.obs[label_key].cat.add_categories("Unknown")
adata_ref.obs[label_key] = adata_ref.obs[label_key].fillna("Unknown")
adata_ref.obs[label_key] = adata_ref.obs[label_key].astype(str)
adata_ref.obs[label_key] = adata_ref.obs[label_key].astype("category")

print(f"NaNs remaining: {adata_ref.obs[label_key].isna().sum()}")

## UMAP + Plot

In [ ]:
plot_colors = [batch_key, label_key]

for emb_key in embedding_obsm_keys:
    if emb_key in adata_ref.obsm.keys():
        compute_and_plot_umap(
            adata=adata_ref,
            embedding_key=emb_key,
            color_keys=plot_colors,
            output_dir=OUTPUT_DIR,
            palette_map=CELL_TYPE_PALETTE
        )
    else:
        print(f"Skipping UMAP for {emb_key}: key not found in obsm.")